In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp
import ROOT
from ROOT import TMVA
import pickle

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)


from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.makedf import make_cc1pidf
from analysis_village.cc1pi.DataFrameUtils import DFCleaning as DFUtils
from analysis_village.cc1pi.DataFrameUtils.Files import filename_to_dataframe
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.Optimize import OptimizationUtils
from analysis_village.cc1pi.Optimize import ConfusionMatricesUtils
from analysis_village.cc1pi.BDTs import BDTTrainingUtils

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

np.seterr(divide='ignore', invalid='ignore', over='ignore')


In [ ]:
## Check keys in each file
optimization_file = "/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_1e20_training.df"
development_sample_file = "/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV.df"
print("keys in test_file")
splh.print_keys(optimization_file)

## Check split multiplicity
print("mc_bnb_cosmic_file n_split: %d" %splh.get_n_split(optimization_file))

In [ ]:
## Define keys to load
print('MC dataframes')
n_max_concat = 12 ## for big files, each key could have more than one split
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
optimization_df = splh.load_dfs(optimization_file, keys2load, n_max_concat)
development_sample_df = splh.load_dfs(development_sample_file, keys2load, n_max_concat)
print('test data loaded!')

In [ ]:
#Perform duplication validation
print("duplication for Spring Production BNB + Cosmic sample")
DFUtils.find_duplicate_run_evt_combinations(optimization_df['hdr'])
DFUtils.find_duplicate_run_evt_combinations(development_sample_df['hdr'])


In [ ]:
DFUtils.plot_duplicate_run_subrun_evt_distribution(optimization_df["hdr"], "test_df")
DFUtils.plot_duplicate_run_subrun_evt_distribution(development_sample_df["hdr"], "dev_sample_df")

In [ ]:
### Filter the hdr DataFrame first, then filter other DataFrames by matching with the hdr DataFrame
optimization_df["hdr"] = DFUtils.filter_unique_events(optimization_df["hdr"])
DFUtils.find_duplicate_run_evt_combinations(optimization_df["hdr"])
#filter the rest of dataframe keys
for key in keys2load:
    if key == "hdr":
        continue
    optimization_df[key] = DFUtils.filter_using_hdr(optimization_df[key], optimization_df["hdr"])


development_sample_df["hdr"] = DFUtils.filter_unique_events(development_sample_df["hdr"])
DFUtils.find_duplicate_run_evt_combinations(development_sample_df["hdr"])
#filter the rest of dataframe keys
for key in keys2load:
    if key == "hdr":
        continue
    development_sample_df[key] = DFUtils.filter_using_hdr(development_sample_df[key], development_sample_df["hdr"])    

In [ ]:
SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]

In [ ]:
def get_n_evt(df):
    unique_count = df.index.droplevel(
        list(df.index.names[2:])  # drop everything except first two levels
    ).nunique()
    return unique_count

In [ ]:
## Collect pot scale for MC
mc_tot_pot = optimization_df["hdr"]['pot'].sum()
#mc_low_th_tot_pot = mc_rockbox_th1to100_dfs["hdr"]['pot'].sum()

data_tot_pot = 1e20
#data_tot_pot = data_bnb_light_dfs["hdr"]['pot'].sum()
#data_tot_TOR860 = data_bnb_light_dfs["pot"]['TOR860'].sum()
#data_tot_TOR875 = data_bnb_light_dfs["pot"]['TOR875'].sum()

print("mc_tot_pot: %e" %(mc_tot_pot))
#print("mc_low_thtot_pot: %e" %(mc_low_th_tot_pot))

#print("data_tot_pot: %e" %(data_tot_pot))
#print("data_tot_TOR860: %e" %(data_tot_TOR860))
#print("data_tot_TOR875: %e" %(data_tot_TOR875))

target_pot = data_tot_pot
mc_pot_scale = target_pot / mc_tot_pot
#mc_low_th_scale = target_pot / mc_low_th_tot_pot
print("MC POT scale: %.3f" %(mc_pot_scale))
#print("MC Low Th. POT scale: %.3f" %(mc_low_th_scale))

In [ ]:
## Comparison between observed and expected total number of recorded spills
n_evt_mc = get_n_evt(optimization_df["hdr"])
#n_evt_mc_low_th = get_n_evt(mc_rockbox_th1to100_dfs["hdr"])

#print("n_evt_data_onbeam: %d" %n_record_spill_data)
#print("n_evt_exp.: %f" %(n_evt_mc * mc_pot_scale + n_evt_mc_low_th * mc_low_th_scale +n_record_spill_offbeam_data * intime_gate_scale))
print("- n_evt_mc: %f" %(n_evt_mc * mc_pot_scale))
#print("- n_evt_mc_low_th: %f" %(n_evt_mc_low_th * mc_low_th_scale))
#print("- n_evt_data_offbeam: %f" %(n_record_spill_offbeam_data * intime_gate_scale))

In [ ]:
evt_df = optimization_df['cc1pi']
hdr_df = optimization_df['hdr']
nu_df = optimization_df['nudf']

dev_sample_evt_df = development_sample_df['cc1pi']
dev_sample_nu_df = development_sample_df['nudf']

In [ ]:
new_columns = []
for c in nu_df.columns:
    new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
nu_df.columns = pd.MultiIndex.from_tuples(new_columns)

new_columns = []
for c in dev_sample_nu_df.columns:
    new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
dev_sample_nu_df.columns = pd.MultiIndex.from_tuples(new_columns)

In [ ]:
matchdf = ph.multicol_merge(evt_df.reset_index(), nu_df.reset_index(),
                            left_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("slc", "tmatch","idx","","","")],
                            right_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("rec.mc.nu..index", "","","","","")], 
                            how="left") ## -- save all sllices
#Reindex so it is again "__ntuple","entry", "slice_id"
matchdf = matchdf.set_index(evt_df.index.names, verify_integrity=True)
#Remove "rec.mc.nu..index"
matchdf = matchdf.drop(columns=[('rec.mc.nu..index','','','','','')])
matchdf.loc[:, ('truth', 'nu_categ','','','','')] = (
    matchdf.loc[:, ('truth', 'nu_categ','','','','')].fillna('cosmic')
)

dev_sample_matchdf = ph.multicol_merge(dev_sample_evt_df.reset_index(), dev_sample_nu_df.reset_index(),
                            left_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("slc", "tmatch","idx","","","")],
                            right_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("rec.mc.nu..index", "","","","","")], 
                            how="left") ## -- save all sllices
#Reindex so it is again "__ntuple","entry", "slice_id"
dev_sample_matchdf = dev_sample_matchdf.set_index(dev_sample_evt_df.index.names, verify_integrity=True)
#Remove "rec.mc.nu..index"
dev_sample_matchdf = dev_sample_matchdf.drop(columns=[('rec.mc.nu..index','','','','','')])
dev_sample_matchdf.loc[:, ('truth', 'nu_categ','','','','')] = (
    dev_sample_matchdf.loc[:, ('truth', 'nu_categ','','','','')].fillna('cosmic')
)

In [ ]:
evt_df = matchdf
dev_sample_evt_df = dev_sample_matchdf

# BDT Training Start

In [ ]:

file_dir = "/exp/sbnd/data/users/lpelegri/Graphs/ProtonBDT"
os.makedirs(file_dir, exist_ok=True)  # create directory if needed

In [ ]:
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_chi2_exp_pol  = ('pfp', 'trk', 'chi2_exp_pol', '', '', '')
col_frac_50 = ('pfp', 'trk', 'frac50', '', '', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')
BDT_columns = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_frac_50]

cut_mask = CutMasks.t0_cut_mask(evt_df)  & CutMasks.nu_score_cut_mask(evt_df)  & (evt_df.truth.nu_categ != "cosmic")

#Define the track df
signal_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 2212)  & (evt_df.pfp.trk.truth.p.end_process == 7 ) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

#Not exiting for training
bkg_df = bkg_df[bkg_df.pfp.is_exiting == False]
signal_df = signal_df[signal_df.pfp.is_exiting == False]

#Only with good values
bdt_mask_signal = BDTTrainingUtils.bdt_quality_mask(signal_df, BDT_columns)
bdt_mask_bkg    = BDTTrainingUtils.bdt_quality_mask(bkg_df, BDT_columns)
signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]

#MIP mask
signal_df = signal_df[CutMasks.is_MIP_candidate_mask(signal_df)]
bkg_df    = bkg_df[CutMasks.is_MIP_candidate_mask(bkg_df)]

print(CTE.MIP_candidate_min_TL)
print(CTE.MIP_candidate_max_muon_score)
print(CTE.MIP_candidate_min_proton_score)
print(len(signal_df))
print(len(bkg_df))

In [ ]:
BDT_columns_normal = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_frac_50]
#BDT_columns_3vars = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol_3var, col_frac_50]

model_vec_normal = BDTTrainingUtils.create_models_for_columns(BDT_columns_normal)
#model_vec_3vars = create_models_for_columns(BDT_columns_3vars)

In [ ]:

# Train all models
trained_models, X_train_normal, X_test_normal, y_train_normal, y_test_normal = BDTTrainingUtils.train_all_models(
    signal_df,
    bkg_df,
    BDT_columns_normal,
    model_vec_normal,
    test_size=0.2
)
'''
# Train all models
X_train_3vars, X_test_3vars, y_train_3vars, y_test_3vars = train_all_models(
    signal_df,
    bkg_df,
    BDT_columns_3vars,
    model_vec_3vars,
    test_size=0.2
)
'''

In [ ]:
xgb_scores_train, xgb_scores_test   = BDTTrainingUtils.get_scores(trained_models["XGB"], X_train_normal, X_test_normal)
bdt_scores_train, bdt_scores_test   = BDTTrainingUtils.get_scores(trained_models["BDT"], X_train_normal, X_test_normal)
bdtg_scores_train, bdtg_scores_test = BDTTrainingUtils.get_scores(trained_models["BDTG"], X_train_normal, X_test_normal)
bdtb_scores_train, bdtb_scores_test = BDTTrainingUtils.get_scores(trained_models["BDTB"], X_train_normal, X_test_normal)
random_forest_scores_train, random_forest_scores_test = BDTTrainingUtils.get_scores(trained_models["RF"], X_train_normal, X_test_normal)
print(len(bdtb_scores_train))
print(len(bdtb_scores_test))

In [ ]:
plt.figure(figsize=(6,6))
BDTTrainingUtils.plot_roc(y_test_normal, bdt_scores_test, "BDT")
BDTTrainingUtils.plot_roc(y_test_normal, bdtg_scores_test, "BDTG")
BDTTrainingUtils.plot_roc(y_test_normal, bdtb_scores_test, "BDTB")
BDTTrainingUtils.plot_roc(y_test_normal, bdtb_scores_test, "XGB")
BDTTrainingUtils.plot_roc(y_test_normal, random_forest_scores_test, "RandomForest")
plt.plot([0,1],[0,1], "k--")
plt.xlabel("Background efficiency")
plt.ylabel("Signal efficiency")
plt.legend()
plt.show()

In [ ]:
'''
sig_train = random_forest_scores_train[y_train==1]
sig_test  = random_forest_scores_test[y_test==1]
bkg_train = random_forest_scores_train[y_train==0]
bkg_test  = random_forest_scores_test[y_test==0]

'''

sig_train_bdtg = bdtg_scores_train[y_train_normal==1]
sig_test_bdtg  = bdtg_scores_test[y_test_normal==1]
bkg_train_bdtg = bdtg_scores_train[y_train_normal==0]
bkg_test_bdtg  = bdtg_scores_test[y_test_normal==0]

sig_train_bdt = bdt_scores_train[y_train_normal==1]
sig_test_bdt  = bdt_scores_test[y_test_normal==1]
bkg_train_bdt = bdt_scores_train[y_train_normal==0]
bkg_test_bdt  = bdt_scores_test[y_test_normal==0]


sig_train_xgb = xgb_scores_train[y_train_normal==1]
sig_test_xgb = xgb_scores_test[y_test_normal==1]
bkg_train_xgb = xgb_scores_train[y_train_normal==0]
bkg_test_xgb = xgb_scores_test[y_test_normal==0]

In [ ]:
'''
df_imp = BDTTrainingUtils.plot_transformed_importance(trained_models["BDT"], BDT_columns_normal)
df_imp = BDTTrainingUtils.plot_transformed_importance(trained_models["BDTG"], BDT_columns_normal)
df_imp = BDTTrainingUtils.plot_transformed_importance(trained_models["XGB"], BDT_columns_normal)
'''
df_imp = BDTTrainingUtils.plot_importance_no_transform(trained_models["BDT"], BDT_columns_normal)
df_imp = BDTTrainingUtils.plot_importance_no_transform(trained_models["BDTG"], BDT_columns_normal)
df_imp = BDTTrainingUtils.plot_importance_no_transform(trained_models["XGB"], BDT_columns_normal)

In [ ]:
corr_matrix = BDTTrainingUtils.plot_correlation_matrix(signal_df, BDT_columns_normal, title="Input variable correlations")
corr_matrix = BDTTrainingUtils.plot_correlation_matrix(bkg_df, BDT_columns_normal, title="Input variable correlations")

In [ ]:
# Plot
print(len(sig_train_bdt))
print(len(sig_test_bdt))
print(len(bkg_train_bdt))
print(len(bkg_test_bdt))
BDTTrainingUtils.plot_response(sig_train_bdt, sig_test_bdt, bkg_train_bdt, bkg_test_bdt, "BDT Response")
BDTTrainingUtils.plot_response(sig_train_bdtg, sig_test_bdtg, bkg_train_bdtg, bkg_test_bdtg, "BDTG Response")
BDTTrainingUtils.plot_response(sig_train_xgb, sig_test_xgb, bkg_train_xgb, bkg_test_xgb, "XGB Response")

In [ ]:
with open("test_bdts/bdt_model_proton_v2.pkl", "wb") as f:
    pickle.dump(trained_models["BDT"], f)

with open("test_bdts/bdtg_model_proton.pkl", "wb") as f:
    pickle.dump(trained_models["BDTG"], f)
    
with open("test_bdts/xgb_model.pkl", "wb") as f:
    pickle.dump(trained_models["XGB"], f)
    
'''
with open("bdtb_model.pkl", "wb") as f:
    pickle.dump(bdtb, f)

with open("random_forest.pkl", "wb") as f:
    pickle.dump(rf, f)
'''

In [ ]:
evt_df = matchdf
dev_sample_evt_df = dev_sample_matchdf

BDT_input_columns = [
    ('pfp','trk','chi2pid','best','chi2_muon',''),
    ('pfp','trk','chi2pid','best','chi2_proton',''),
    ('pfp','trk','chi2_exp_pol','','',''),
    ('pfp','trk','frac50','','','')
]

col_BDT_score_proton = ('pfp','BDT_score_proton','','','','')
col_BDTG_score_proton = ('pfp','BDTG_score_proton','','','','')
col_XGB_score_proton = ('pfp','XGB_score_proton','','','','')

evt_df = make_cc1pidf.add_bdt_score(
    evt_df,
    "test_bdts/bdt_model_proton_v2.pkl",
    BDT_input_columns,
    col_BDT_score_proton
)

evt_df = make_cc1pidf.add_bdt_score(
    evt_df,
    #"test_bdts/bdtg_model_proton.pkl",
    "bdtg_model_proton.pkl",
    BDT_input_columns,
    col_BDTG_score_proton
)

evt_df = make_cc1pidf.add_bdt_score(
    evt_df,
    "test_bdts/xgb_model.pkl",
    BDT_input_columns,
    col_XGB_score_proton
)

dev_sample_evt_df = make_cc1pidf.add_bdt_score(
    dev_sample_evt_df,
    "test_bdts/bdt_model_proton_v2.pkl",
    BDT_input_columns,
    col_BDT_score_proton
)

dev_sample_evt_df = make_cc1pidf.add_bdt_score(
    dev_sample_evt_df,
    "bdtg_model_proton.pkl",
    BDT_input_columns,
    col_BDTG_score_proton
)

dev_sample_evt_df = make_cc1pidf.add_bdt_score(
    dev_sample_evt_df,
    "test_bdts/xgb_model.pkl",
    BDT_input_columns,
    col_XGB_score_proton
)

In [ ]:
'''
og_xmlWeightsFile = "/exp/sbnd/data/users/lpelegri/cc1pi/CAF_analisis/UtilsCutsAndVars/BDT_proton_weights.xml"
retrain_xmlWeightsFile = "/home/lpelegri/cafpyana/analysis_village/cc1pi/BDTs/cafpyana_proton_TMVAClassification_BDT.weights.xml"

og_output_column = ('pfp','trk','og_bdt_proton_score','','','')
retrain_output_column = ('pfp','trk','retrain_bdt_proton_score','','','')

evt_df = make_cc1pidf.add_BDT_TMVA_proton_column(evt_df, og_xmlWeightsFile, og_output_column)
dev_sample_evt_df = make_cc1pidf.add_BDT_TMVA_proton_column(dev_sample_evt_df, og_xmlWeightsFile, og_output_column)

evt_df = make_cc1pidf.add_BDT_TMVA_proton_column(evt_df, retrain_xmlWeightsFile, retrain_output_column)
dev_sample_evt_df = make_cc1pidf.add_BDT_TMVA_proton_column(dev_sample_evt_df, retrain_xmlWeightsFile, retrain_output_column)
'''

In [ ]:
evt_df.columns

In [ ]:
cut_mask = CutMasks.t0_cut_mask(dev_sample_evt_df)  &  CutMasks.nu_score_cut_mask(dev_sample_evt_df)  & CutMasks.track_cut_mask(dev_sample_evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(dev_sample_evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(dev_sample_evt_df,SLICE_LEVELS)
plot_df = dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & (dev_sample_evt_df.pfp.is_exiting == False)].copy()

config_BDT_proton_score = HistogramConfig(
    data_column=('pfp','BDT_score_proton','','','',''),
    bins=np.linspace(-0.5, 1.1, 41),
    xlabel='BDT proton score',
    ylabel='Entries',
    title='BDT proton Score'
)

config_BDTG_proton_score = HistogramConfig(
    data_column=('pfp','BDTG_score_proton','','','',''),
    bins=np.linspace(-4, 8, 41),
    xlabel='BDTG proton score',
    ylabel='Entries',
    title='BDTG proton Score'
)

'''
config_BDT_usual_proton_score = HistogramConfig(
    data_column=('pfp','trk','og_bdt_proton_score','','',''),
    bins=np.linspace(-0.4, 0.5, 41),
    xlabel='TMVA OG BDT proton score',
    ylabel='Entries',
    title='TMVA OG BDT proton Score'
)

config_BDT_retrain_proton_score = HistogramConfig(
    data_column=('pfp','trk','retrain_bdt_proton_score','','',''),
    bins=np.linspace(-0.4, 0.5, 41),
    xlabel='TMVA retrain BDT proton score',
    ylabel='Entries',
    title='TMVA retrain BDT proton Score'
)
'''

config_BDT_XGB_proton_score = HistogramConfig(
    data_column=('pfp','XGB_score_proton','','','',''),
    bins=np.linspace(0, 1, 41),
    xlabel='XGB proton score',
    ylabel='Entries',
    title='XGB proton Score'
)


plot_stacked_histogram(
    plot_df,
    config=config_BDT_proton_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)

plot_stacked_histogram(
    plot_df,
    config=config_BDTG_proton_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)

'''
plot_stacked_histogram(
    plot_df,
    config=config_BDT_usual_proton_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)

plot_stacked_histogram(
    plot_df,
    config=config_BDT_retrain_proton_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)
'''

plot_stacked_histogram(
    plot_df,
    config=config_BDT_XGB_proton_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)


In [ ]:
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_chi2_exp_pol  = ('pfp', 'trk', 'chi2_exp_pol', '', '', '')
col_frac_50 = ('pfp', 'trk', 'frac50', '', '', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')

cut_mask = CutMasks.t0_cut_mask(evt_df) & CutMasks.nu_score_cut_mask(evt_df)  & CutMasks.track_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(evt_df,SLICE_LEVELS)


'''
#Define the track df
bkg_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

signal_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 2212)  & (evt_df.pfp.trk.truth.p.end_process == 7 ) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]
'''

#Define the track df
signal_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 2212)  & (evt_df.pfp.trk.truth.p.end_process == 7 ) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

#Not exiting for training
bkg_df = bkg_df[bkg_df.pfp.is_exiting == False]
signal_df = signal_df[signal_df.pfp.is_exiting == False]

#Only with good values
bdt_mask_signal = BDTTrainingUtils.bdt_quality_mask(signal_df, BDT_columns)
bdt_mask_bkg    = BDTTrainingUtils.bdt_quality_mask(bkg_df, BDT_columns)
signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]

#MIP mask
signal_df = signal_df[CutMasks.is_MIP_candidate_mask(signal_df)]
bkg_df    = bkg_df[CutMasks.is_MIP_candidate_mask(bkg_df)]

In [ ]:

col_BDT_score_proton = ('pfp','BDT_score_proton','','','','')
col_BDTG_score_proton = ('pfp','BDTG_score_proton','','','','')
col_XGB_score_proton = ('pfp','XGB_score_proton','','','','')
col_BDT_TMVA_og_score_proton = ('pfp','trk','og_bdt_proton_score','','','')
col_BDT_TMVA_retrain_score_proton = ('pfp','trk','retrain_bdt_proton_score','','','')

sing = ">"
bdt_opt_len, best_bdt, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_BDT_score_proton,
    cut_type=sing,
    xlabel="length",
    title="BDT optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-0.6, 1.2),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0, # <-- new parameter
)

bdtg_opt_len, best_bdtg, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_BDTG_score_proton,
    cut_type=sing,
    xlabel="length",
    title="BDTG optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-4, 8),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0, # <-- new parameter
)
'''

TMV_og_bdt_df, best_TMVA_og = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_BDT_TMVA_og_score_proton,
    cut_type=sing,
    xlabel="length",
    title="TMVA OG",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-0.4, 0.5),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0, # <-- new parameter
)

TMV_retrain_bdt_df, best_TMVA_retrain = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_BDT_TMVA_retrain_score_proton,
    cut_type=sing,
    xlabel="length",
    title="TMVA retrain",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-0.4, 0.5),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0, # <-- new parameter
)
'''


xgb_bdt_df, best_xgb, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_XGB_score_proton,
    cut_type=sing,
    xlabel="length",
    title="XGB optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(0, 1),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0, # <-- new parameter
)

In [ ]:

col_BDT_score_proton = ('pfp','BDT_score_proton','','','','')
col_BDTG_score_proton = ('pfp','BDTG_score_proton','','','','')
col_XGB_score_proton = ('pfp','XGB_score_proton','','','','')
col_BDT_TMVA_og_score_proton = ('pfp','trk','og_bdt_proton_score','','','')
col_BDT_TMVA_retrain_score_proton = ('pfp','trk','retrain_bdt_proton_score','','','')

sing = ">"
bdt_opt_len, best_bdt, fig = OptimizationUtils.optimize_cut_accuracy(
    signal_df,
    bkg_df,
    column= col_BDT_score_proton,
    cut_type=sing,
    xlabel="length",
    title="BDT optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-0.6, 1.2),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False
)

bdtg_opt_len, best_bdtg, fig = OptimizationUtils.optimize_cut_accuracy(
    signal_df,
    bkg_df,
    column= col_BDTG_score_proton,
    cut_type=sing,
    xlabel="length",
    title="BDTG optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-4, 8),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False
)
'''
TMV_og_bdt_df, best_TMVA_og = OptimizationUtils.optimize_cut_accuracy(
    signal_df,
    bkg_df,
    column= col_BDT_TMVA_og_score_proton,
    cut_type=sing,
    xlabel="length",
    title="TMVA OG",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-0.4, 0.5),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0, # <-- new parameter
)

TMV_retrain_bdt_df, best_TMVA_retrain = OptimizationUtils.optimize_cut_accuracy(
    signal_df,
    bkg_df,
    column= col_BDT_TMVA_retrain_score_proton,
    cut_type=sing,
    xlabel="length",
    title="TMVA retrain",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(-0.4, 0.5),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0, # <-- new parameter
)

'''

xgb_bdt_df, best_xgb, fig = OptimizationUtils.optimize_cut_accuracy(
    signal_df,
    bkg_df,
    column= col_XGB_score_proton,
    cut_type=sing,
    xlabel="length",
    title="XGB optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="proton",
    xlim=(0, 1),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False
)

In [ ]:
SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]
def proton_test_BDT_cut_mask(df, group_levels, column, cut):
    BDT_proton_df = df[(CutMasks.is_MIP_candidate_mask(df)) & (df[column] > cut)]
    
    # Count how many pfps per slice
    candidate_counts = BDT_proton_df.groupby(level=group_levels).size()
 
    # Get only slices with at least 2 pfps
    valid_slices = candidate_counts[candidate_counts == 2].index

    # Apply the mask to original DataFrame
    final_mask = pd.Series(df.index.droplevel('rec.slc.reco.pfp..index').isin(valid_slices), index=df.index)

    return final_mask

In [ ]:
cut_mask = CutMasks.t0_cut_mask(dev_sample_evt_df) & CutMasks.nu_score_cut_mask(dev_sample_evt_df)  & CutMasks.track_cut_mask(dev_sample_evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(dev_sample_evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(dev_sample_evt_df,SLICE_LEVELS)
print("control")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask], target_categ="CC1pi")
'''
print("TMVA og")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask  & proton_test_BDT_cut_mask(dev_sample_evt_df, SLICE_LEVELS, col_BDT_TMVA_og_score_proton, best_TMVA_og['cut'])], target_categ="CC1pi")
print("TMVA retrain")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask  & proton_test_BDT_cut_mask(dev_sample_evt_df, SLICE_LEVELS, col_BDT_TMVA_retrain_score_proton, best_TMVA_retrain['cut'])], target_categ="CC1pi")
'''
print("BDT")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask & proton_test_BDT_cut_mask(dev_sample_evt_df, SLICE_LEVELS, col_BDT_score_proton, best_bdt['cut'])], target_categ="CC1pi")
print("BDTG")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask & proton_test_BDT_cut_mask(dev_sample_evt_df, SLICE_LEVELS, col_BDTG_score_proton, best_bdtg['cut'])], target_categ="CC1pi")
print("XGB")
HelperFunctions.print_category_metrics(dev_sample_evt_df, dev_sample_evt_df[cut_mask & proton_test_BDT_cut_mask(dev_sample_evt_df, SLICE_LEVELS, col_XGB_score_proton, best_xgb['cut'])], target_categ="CC1pi")

In [ ]:
def build_pid_confusion_matrix(df, bdtg_threshold=0.5, key=('pfp', 'BDTG_score_proton', '', '', '', ''), weight_key=None):
    """
    Builds a confusion matrix for Proton vs Muon/Pion identification.
    
    True Labels: [Muon/Pion, Proton]
    Reco Labels: [Reco Muon/Pion, Reco Proton]
    """
    import numpy as np
    import pandas as pd

    # 1. Define Keys
    pdg_key = ('pfp', 'trk', 'truth', 'p', 'pdg', '')
    
    # 2. Filter for specific particles only (13, 211, 2212)
    # We use .copy() to avoid SettingWithCopy warnings
    mask_valid = df[pdg_key].abs().isin([13, 211, 2212])
    df_filtered = df[mask_valid].copy()
    
    if df_filtered.empty:
        print("Warning: No valid PDGs (13, 211, 2212) found in DataFrame.")
        return np.zeros((2,2)), [], []

    # 3. Assign True Category (0: Muon/Pion, 1: Proton)
    df_filtered['true_pid'] = np.where(df_filtered[pdg_key].abs() == 2212, 1, 0)
    
    # 4. Assign Reco Category (1: Proton if < threshold, 0: Muon/Pion)
    # Logic: Score < threshold -> Reco Proton
    df_filtered['reco_pid'] = np.where(df_filtered[key] < bdtg_threshold, 1, 0)
    
    # 5. Define Weights
    # If no weight is provided, count each row as 1.0
    if weight_key is None:
        weights = np.ones(len(df_filtered))
    else:
        weights = df_filtered[weight_key]

    # 6. Build Matrix using pd.crosstab (much safer than groupby.sum for CMs)
    # This creates the matrix directly with reco on index and true on columns
    cm_df = pd.crosstab(
        df_filtered['reco_pid'], 
        df_filtered['true_pid'], 
        values=weights, 
        aggfunc='sum'
    ).fillna(0)

    # Ensure the matrix is 2x2 even if a category is missing
    cm = np.zeros((2, 2))
    for r in [0, 1]:
        for t in [0, 1]:
            if r in cm_df.index and t in cm_df.columns:
                cm[r, t] = cm_df.loc[r, t]

    # Labels
    x_labels = [r"True $\mu/\pi$", "True Proton"]
    y_labels = [r"Reco $\mu/\pi$", "Reco Proton"]
    
    return cm, x_labels, y_labels

In [ ]:

from analysis_village.cc1pi.Optimize import ConfusionMatricesUtils

cut_mask = CutMasks.t0_cut_mask(evt_df) & CutMasks.nu_score_cut_mask(evt_df)  & CutMasks.track_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(evt_df,SLICE_LEVELS)
cm_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)|(abs(evt_df.pfp.trk.truth.p.pdg) == 2212)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

#Not exiting for training
cm_df = cm_df[cm_df.pfp.is_exiting == False]
cm_df = cm_df[CutMasks.is_MIP_candidate_mask(cm_df)]
cm, x_labs, y_labs = build_pid_confusion_matrix(cm_df, bdtg_threshold = best_bdtg['cut'], key = ('pfp', 'BDTG_score_proton', '', '', '', ''))
cm_plot = ConfusionMatricesUtils.plot_confusion_matrix(cm, x_labs, y_labs)
cm_plot.savefig(file_dir + "/confusion_matrix_contained_p", dpi=300)
plt.show()


cut_mask = CutMasks.t0_cut_mask(evt_df) & CutMasks.nu_score_cut_mask(evt_df)  & CutMasks.track_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(evt_df,SLICE_LEVELS)
cm_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)|(abs(evt_df.pfp.trk.truth.p.pdg) == 2212)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]
#Exiting for training
cm_df = cm_df[cm_df.pfp.is_exiting == True]
cm_df = cm_df[CutMasks.is_MIP_candidate_mask(cm_df)]
cm, x_labs, y_labs = build_pid_confusion_matrix(cm_df, bdtg_threshold = best_bdtg['cut'], key = ('pfp', 'BDTG_score_proton', '', '', '', ''))
cm_plot = ConfusionMatricesUtils.plot_confusion_matrix(cm, x_labs, y_labs)
cm_plot.savefig(file_dir + "/confusion_matrix_exiting_p", dpi=300)
plt.show()


cut_mask = CutMasks.t0_cut_mask(evt_df) & CutMasks.nu_score_cut_mask(evt_df)  & CutMasks.track_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(evt_df,SLICE_LEVELS)
cm_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)|(abs(evt_df.pfp.trk.truth.p.pdg) == 2212)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]
#ALL for training
cm_df = cm_df[CutMasks.is_MIP_candidate_mask(cm_df)]
cm, x_labs, y_labs = build_pid_confusion_matrix(cm_df, bdtg_threshold = best_bdtg['cut'], key = ('pfp', 'BDTG_score_proton', '', '', '', ''))
cm_plot = ConfusionMatricesUtils.plot_confusion_matrix(cm, x_labs, y_labs)
cm_plot.savefig(file_dir + "/confusion_matrix_all_p", dpi=300)
plt.show()

# Choose your threshold

# Training for µ/π separation

In [ ]:


col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')
col_chi2_exp_pol  = ('pfp', 'trk', 'chi2_exp_pol', '', '', '')
col_chi2_exp_pol_3var = ('pfp', 'trk', 'chi2_exp_pol_3var', '', '', '')
col_frac_50 = ('pfp', 'trk', 'frac50', '', '', '')
col_scatter_angle_ratio = ('pfp', 'scatter_angle_ratio', '', '', '', '')
col_max_daughter_hits = ('pfp', 'max_daughter_hits', '', '', '', '')

cut_mask = CutMasks.t0_cut_mask(evt_df)  & CutMasks.nu_score_cut_mask(evt_df)  & (evt_df.truth.nu_categ != "cosmic")
BDT_columns_mupi = [
    col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_scatter_angle_ratio, col_max_daughter_hits
]


#Define the track df
signal_df = evt_df[
    cut_mask &
    (abs(evt_df.pfp.trk.truth.p.pdg) == 13) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 211) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

#Not exiting for training
bkg_df = bkg_df[bkg_df.pfp.is_exiting == False]
signal_df = signal_df[signal_df.pfp.is_exiting == False]

#Only with good values
bdt_mask_signal = BDTTrainingUtils.bdt_quality_mask(signal_df, BDT_columns_mupi)
bdt_mask_bkg    = BDTTrainingUtils.bdt_quality_mask(bkg_df, BDT_columns_mupi)
signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]

#MIP mask
signal_df = signal_df[CutMasks.is_MIP_candidate_mask(signal_df)]
bkg_df    = bkg_df[CutMasks.is_MIP_candidate_mask(bkg_df)]

print(len(signal_df))
print(len(bkg_df))


In [ ]:
model_vec_mupi =BDTTrainingUtils. create_models_for_columns(BDT_columns_mupi)
trained_models_mupi, X_train_mupi, X_test_mupi, y_train_mupi, y_test_mupi = BDTTrainingUtils.train_all_models(
    signal_df,
    bkg_df,
    BDT_columns_mupi,
    model_vec_mupi,
    test_size=0.2
)

In [ ]:
bdt_scores_mupi_train, bdt_scores_mupi_test   = BDTTrainingUtils.get_scores(trained_models_mupi["BDT"], X_train_mupi, X_test_mupi)
bdtg_scores_mupi_train, bdtg_scores_mupi_test   = BDTTrainingUtils.get_scores(trained_models_mupi["BDTG"], X_train_mupi, X_test_mupi)
bdtb_scores_mupi_train, bdtb_scores_mupi_test   = BDTTrainingUtils.get_scores(trained_models_mupi["BDTB"], X_train_mupi, X_test_mupi)
rf_scores_mupi_train, rf_scores_mupi_test   = BDTTrainingUtils.get_scores(trained_models_mupi["RF"], X_train_mupi, X_test_mupi)
xgb_scores_mupi_train, xgb_scores_mupi_test   = BDTTrainingUtils.get_scores(trained_models_mupi["XGB"], X_train_mupi, X_test_mupi)

plt.figure(figsize=(6,6))
BDTTrainingUtils.plot_roc(y_test_mupi, bdt_scores_mupi_test, "BDT")
BDTTrainingUtils.plot_roc(y_test_mupi, bdtg_scores_mupi_test, "BDTG")
BDTTrainingUtils.plot_roc(y_test_mupi, bdtb_scores_mupi_test, "BDTB")
BDTTrainingUtils.plot_roc(y_test_mupi, rf_scores_mupi_test, "RF")
BDTTrainingUtils.plot_roc(y_test_normal, bdtb_scores_test, "XGB")
plt.plot([0,1],[0,1], "k--")
plt.xlabel("Background efficiency")
plt.ylabel("Signal efficiency")
plt.legend()
plt.show()



sig_train = bdt_scores_mupi_train[y_train_mupi==1]
sig_test  = bdt_scores_mupi_test[y_test_mupi==1]
bkg_train = bdt_scores_mupi_train[y_train_mupi==0]
bkg_test  = bdt_scores_mupi_test[y_test_mupi==0]
BDTTrainingUtils.plot_response(sig_train, sig_test, bkg_train, bkg_test, "BDT Response")

sig_train = bdtg_scores_mupi_train[y_train_mupi==1]
sig_test  = bdtg_scores_mupi_test[y_test_mupi==1]
bkg_train = bdtg_scores_mupi_train[y_train_mupi==0]
bkg_test  = bdtg_scores_mupi_test[y_test_mupi==0]
BDTTrainingUtils.plot_response(sig_train, sig_test, bkg_train, bkg_test, "BDTG Response")

sig_train = xgb_scores_mupi_train[y_train_mupi==1]
sig_test  = xgb_scores_mupi_test[y_test_mupi==1]
bkg_train = xgb_scores_mupi_train[y_train_mupi==0]
bkg_test  = xgb_scores_mupi_test[y_test_mupi==0]
BDTTrainingUtils.plot_response(sig_train, sig_test, bkg_train, bkg_test, "XBG Response")


In [ ]:
with open("test_bdts/bdt_model_muon_pion.pkl", "wb") as f:
    pickle.dump(trained_models_mupi["BDT"], f)
with open("test_bdts/bdtg_model_muon_pion.pkl", "wb") as f:
    pickle.dump(trained_models_mupi["BDTG"], f)
with open("test_bdts/xgb_model_muon_pion.pkl", "wb") as f:
    pickle.dump(trained_models_mupi["XGB"], f)

In [ ]:
dev_sample_evt_df = dev_sample_matchdf

BDT_input_columns = [
    col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_scatter_angle_ratio, col_max_daughter_hits
]

col_BDT_score_muon_pion= ('pfp','BDT_score_muon_pion','','','','')

dev_sample_evt_df = make_cc1pidf.add_bdt_score(
    dev_sample_evt_df,
    "test_bdts/bdt_model_muon_pion.pkl",
    BDT_input_columns,
    col_BDT_score_muon_pion
)

col_BDTG_score_muon_pion= ('pfp','BDTG_score_muon_pion','','','','')

dev_sample_evt_df = make_cc1pidf.add_bdt_score(
    dev_sample_evt_df,
    "bdtg_model_muon_pion.pkl",
    BDT_input_columns,
    col_BDTG_score_muon_pion
)

col_XGB_score_muon_pion= ('pfp','XGB_score_muon_pion','','','','')

dev_sample_evt_df = make_cc1pidf.add_bdt_score(
    dev_sample_evt_df,
    "test_bdts/xgb_model_muon_pion.pkl",
    BDT_input_columns,
    col_XGB_score_muon_pion
)

In [ ]:
'''
xmlWeightsFile_mp = "/exp/sbnd/data/users/lpelegri/cc1pi/CAF_analisis/UtilsCutsAndVars/BDT_muon_pion_weights.xml"
output_column = ('pfp','trk','og_bdt_muon_pion_score','','','')
dev_sample_evt_df = make_cc1pidf.add_TMVA_BDT_muon_pion_column(dev_sample_evt_df, xmlWeightsFile_mp, output_column)

xmlWeightsFile_mp = "/home/lpelegri/cafpyana/analysis_village/cc1pi/BDTs/cafpyana_muon_pion_TMVAClassification_BDT.weights.xml"
output_column = ('pfp','trk','retrain_bdt_muon_pion_score','','','')
dev_sample_evt_df = make_cc1pidf.add_TMVA_BDT_muon_pion_column(dev_sample_evt_df, xmlWeightsFile_mp, output_column)
'''

In [ ]:
cut_mask = CutMasks.t0_cut_mask(dev_sample_evt_df)  &  CutMasks.nu_score_cut_mask(dev_sample_evt_df)  & CutMasks.track_cut_mask(dev_sample_evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(dev_sample_evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(dev_sample_evt_df,SLICE_LEVELS)
plot_df = dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & (dev_sample_evt_df.pfp.is_exiting == False)]

config_BDT_muon_pion_score = HistogramConfig(
    data_column=('pfp','BDT_score_muon_pion','','','',''),
    bins=np.linspace(-0.6, 1.5, 51),
    xlabel='BDT muon/pion score',
    ylabel='Entries',
    title='BDT muon/pion Score'
)

config_BDTG_muon_pion_score = HistogramConfig(
    data_column=('pfp','BDTG_score_muon_pion','','','',''),
    bins=np.linspace(-6, 6, 51),
    xlabel='BDTG muon/pion score',
    ylabel='Entries',
    title='BDTG muon/pion Score'
)

config_XGB_muon_pion_score = HistogramConfig(
    data_column=('pfp','XGB_score_muon_pion','','','',''),
    bins=np.linspace(0, 1, 51),
    xlabel='XGB muon/pion score',
    ylabel='Entries',
    title='XGB muon/pion Score'
)


'''
config_TMVA_og_BDT_muon_pion_score = HistogramConfig(
    data_column=('pfp','trk','og_bdt_muon_pion_score','','',''),
    bins=np.linspace(-0.25, 0.6, 51),
    xlabel='TMVA og BDT muon/pion score',
    ylabel='Entries',
    title='TMVA BDT muon/pion Score'
)

config_TMVA_retrain_BDT_muon_pion_score = HistogramConfig(
    data_column=('pfp','trk','retrain_bdt_muon_pion_score','','',''),
    bins=np.linspace(-0.25, 0.6, 51),
    xlabel='TMVA retrain BDT muon/pion score',
    ylabel='Entries',
    title='TMVA BDT muon/pion Score'
)
'''
plot_stacked_histogram(
    plot_df,
    config=config_BDT_muon_pion_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)

plot_stacked_histogram(
    plot_df,
    config=config_XGB_muon_pion_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)

plot_stacked_histogram(
    plot_df,
    config=config_BDTG_muon_pion_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)
'''
plot_stacked_histogram(
    plot_df,
    config=config_TMVA_og_BDT_muon_pion_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)

plot_stacked_histogram(
    plot_df,
    config=config_TMVA_retrain_BDT_muon_pion_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)
'''

In [ ]:
def get_muon_pion_slice_mask(df):
    """
    Returns a mask for slices that contain exactly one true muon MIP candidate 
    and exactly one true pion MIP candidate.
    """
    pdg_col = ('pfp', 'trk', 'truth', 'p', 'pdg', '')
    SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]

    # 1. Get the particle-level MIP candidate mask
    # This uses your existing function logic
    mip_particles = CutMasks.is_MIP_candidate_mask(df)

    # 2. Identify True Muons and True Pions that are ALSO MIP candidates
    is_muon_mip = (df[pdg_col].abs() == 13) & mip_particles
    is_pion_mip = (df[pdg_col].abs() == 211) & mip_particles

    # 3. Count these specific candidates per slice
    # transform('sum') broadcasts the count to all rows in the slice
    muon_mip_count = is_muon_mip.groupby(level=SLICE_LEVELS).transform('sum')
    pion_mip_count = is_pion_mip.groupby(level=SLICE_LEVELS).transform('sum')
    
    # 4. Final condition: The slice must have exactly one of each MIP-quality particle
    slice_condition = (muon_mip_count == 1) & (pion_mip_count == 1)
    
    return slice_condition

In [ ]:
def get_contained_slice_mask(df):
    """
    Returns a mask that is True for all particles in a slice ONLY if 
    EVERY particle in that slice has is_exiting == False.
    """
    exit_col = ('pfp', 'is_exiting', '', '', '', '')
    SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]

    # 1. Identify particles that are NOT exiting
    is_contained_particle = (df[exit_col] == False)

    # 2. Group by slice and check if the 'is_contained' condition is True for ALL rows
    # transform('all') broadcasts the result back to the original dataframe shape
    all_contained_mask = is_contained_particle.groupby(level=SLICE_LEVELS).transform('all')

    return all_contained_mask

In [ ]:
import matplotlib.pyplot as plt

# 1. Get the raw numbers
print("BDT")
cm_raw, labels = BDTTrainingUtils.prepare_muon_pion_cm(dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & get_muon_pion_slice_mask(dev_sample_evt_df) & (dev_sample_evt_df.truth.nu_categ == "CC1pi") & get_contained_slice_mask(dev_sample_evt_df)], score_col = ('pfp', 'BDT_score_muon_pion', '', '', '', ''))
ConfusionMatricesUtils.plot_confusion_matrix(cm_raw, labels, cmap=sunset_cmap)

print("BDTG")
cm_raw, labels = BDTTrainingUtils.prepare_muon_pion_cm(dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & get_muon_pion_slice_mask(dev_sample_evt_df) & (dev_sample_evt_df.truth.nu_categ == "CC1pi") & get_contained_slice_mask(dev_sample_evt_df)], score_col = ('pfp','BDTG_score_muon_pion', '', '', '', ''))
ConfusionMatricesUtils.plot_confusion_matrix(cm_raw, labels, cmap=sunset_cmap)


print("XGB")
cm_raw, labels = BDTTrainingUtils.prepare_muon_pion_cm(dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & get_muon_pion_slice_mask(dev_sample_evt_df) & (dev_sample_evt_df.truth.nu_categ == "CC1pi") & get_contained_slice_mask(dev_sample_evt_df)], score_col = ('pfp','XGB_score_muon_pion', '', '', '', ''))
ConfusionMatricesUtils.plot_confusion_matrix(cm_raw, labels, cmap=sunset_cmap)

'''
print("TMVA OG")
cm_raw, labels = BDTTrainingUtils.prepare_muon_pion_cm(dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & get_muon_pion_slice_mask(dev_sample_evt_df) & (dev_sample_evt_df.truth.nu_categ == "CC1pi") & get_contained_slice_mask(dev_sample_evt_df)], score_col = ('pfp','trk', 'og_bdt_muon_pion_score', '', '', ''))
ConfusionMatricesUtils.plot_confusion_matrix(cm_raw, labels, cmap=sunset_cmap)

print("TMVA retrain")
cm_raw, labels = BDTTrainingUtils.prepare_muon_pion_cm(dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & get_muon_pion_slice_mask(dev_sample_evt_df) & (dev_sample_evt_df.truth.nu_categ == "CC1pi") & get_contained_slice_mask(dev_sample_evt_df)], score_col = ('pfp','trk', 'retrain_bdt_muon_pion_score', '', '', ''))
ConfusionMatricesUtils.plot_confusion_matrix(cm_raw, labels, cmap=sunset_cmap)
'''

# Save into root

In [ ]:
import uproot
import pandas as pd
import numpy as np

# Cuts and Columns
chi2_p_cut = 80
chi2_mu_cut = 20
len_cut = 10

col_chi2_mu      = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p       = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_chi2_exp_pol = ('pfp', 'trk', 'chi2_exp_pol', '', '', '')
col_frac_50      = ('pfp', 'trk', 'frac50', '', '', '')
col_len          = ('pfp', 'trk', 'len', '', '', '')

BDT_columns = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_frac_50]

# Initial Cut Mask
cut_mask = CutMasks.t0_cut_mask(evt_df) & CutMasks.nu_score_cut_mask(evt_df) & (evt_df.truth.nu_categ != "cosmic")

# Define signal (Muons/Pions) and background (Protons)
signal_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
].copy()

bkg_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 2212) & (evt_df.pfp.trk.truth.p.end_process == 7) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
].copy()

# Fix: Actually apply the 'Not exiting' filter
signal_df = signal_df[signal_df.pfp.is_exiting == False]
bkg_df = bkg_df[bkg_df.pfp.is_exiting == False]

# Apply BDT quality and PID cuts
bdt_mask_signal = BDTTrainingUtils.bdt_quality_mask(signal_df, BDT_columns)
bdt_mask_bkg    = BDTTrainingUtils.bdt_quality_mask(bkg_df, BDT_columns)

signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]

signal_df = signal_df[(signal_df[col_chi2_mu] < chi2_mu_cut) & (signal_df[col_chi2_p] > chi2_p_cut) & (signal_df[col_len] > len_cut)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < chi2_mu_cut) & (bkg_df[col_chi2_p] > chi2_p_cut) & (bkg_df[col_len] > len_cut)]

# 1. Define mapping and preparation helper
column_mapping = {
    col_chi2_exp_pol: 'chi2_exp_pol0',
    col_frac_50:      'fraction_50_drift_percent',
    col_chi2_p:       'chi2p',
    col_chi2_mu:      'chi2mu'
}

def prepare_for_root_dict(df, mapping):
    """Renames columns and returns a dict of numpy arrays for uproot writing."""
    # Select only the columns needed
    subset = df[list(mapping.keys())].copy()
    # Create dictionary with new names and cast to float64 (double)
    return {mapping[k]: subset[k].values.astype(np.float64) for k in mapping}

# 2. Prepare the data
signal_dict = prepare_for_root_dict(signal_df, column_mapping)
bkg_dict    = prepare_for_root_dict(bkg_df, column_mapping)

# 3. Save to ROOT file
file_path = "/exp/sbnd/data/users/lpelegri/TransferFolder/cafpyana_proton_optimization_trees.root"
with uproot.recreate(file_path) as f:
    f["tree_muonlike"] = signal_dict
    f["tree_proton"]   = bkg_dict

print(f"ROOT file '{file_path}' created successfully.")

In [ ]:
chi2_p_cut = 80
chi2_mu_cut = 20
len_cut = 10
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')

col_chi2_exp_pol  = ('pfp', 'trk', 'chi2_exp_pol', '', '', '')
col_chi2_exp_pol_3var = ('pfp', 'trk', 'chi2_exp_pol_3var', '', '', '')
col_frac_50 = ('pfp', 'trk', 'frac50', '', '', '')
col_scatter_angle_ratio = ('pfp', 'scatter_angle_ratio', '', '', '', '')
col_max_daughter_hits = ('pfp', 'max_daughter_hits', '', '', '', '')
#cut_mask = CutMasks.t0_cut_mask(evt_df)  &  CutMasks.nu_score_cut_mask(evt_df)  & CutMasks.track_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(evt_df,SLICE_LEVELS)
cut_mask = CutMasks.t0_cut_mask(evt_df)  & CutMasks.nu_score_cut_mask(evt_df)  & (evt_df.truth.nu_categ != "cosmic")


#Define the track df
signal_df = evt_df[
    cut_mask &
    (abs(evt_df.pfp.trk.truth.p.pdg) == 13) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 211) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]
#Not exiting for training
bkg_df = bkg_df[bkg_df.pfp.is_exiting == False]
signal_df = signal_df[signal_df.pfp.is_exiting == False]
BDT_columns_mupi = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_scatter_angle_ratio, col_max_daughter_hits]
bdt_mask_signal = bdt_quality_mask(signal_df, BDT_columns_mupi)
bdt_mask_bkg    = bdt_quality_mask(bkg_df, BDT_columns_mupi)
signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]
signal_df = signal_df[(signal_df[col_chi2_mu] < chi2_mu_cut) & (signal_df[col_chi2_p] > chi2_p_cut) & (signal_df[col_len] > len_cut)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < chi2_mu_cut) & (bkg_df[col_chi2_p] > chi2_p_cut) & (bkg_df[col_len] > len_cut)]

# 1. Define mapping and preparation helper
column_mapping = {
    col_frac_50:      'fraction_50_drift_percent',
    col_chi2_p:       'chi2p',
    col_chi2_mu:      'chi2mu',
    col_scatter_angle_ratio: 'track_mcs_scatter_max_ratio',
    col_max_daughter_hits: 'max_daughter_hits'
}
print(signal_df[signal_df.pfp.max_daughter_hits != 0].pfp.max_daughter_hits)
def prepare_for_root_dict(df, mapping):
    """Renames columns and returns a dict of numpy arrays for uproot writing."""
    # Select only the columns needed
    subset = df[list(mapping.keys())].copy()
    # Create dictionary with new names and cast to float64 (double)
    return {mapping[k]: subset[k].values.astype(np.float64) for k in mapping}

# 2. Prepare the data
signal_dict = prepare_for_root_dict(signal_df, column_mapping)
bkg_dict    = prepare_for_root_dict(bkg_df, column_mapping)

# 3. Save to ROOT file
file_path = "/exp/sbnd/data/users/lpelegri/TransferFolder/cafpyana_muon_pion_optimization_trees.root"
with uproot.recreate(file_path) as f:
    f["tree_muon"] = signal_dict
    f["tree_pion"]   = bkg_dict

print(f"ROOT file '{file_path}' created successfully.")

In [ ]:
print(signal_df[signal_df.pfp.max_daughter_hits != 0].pfp.max_daughter_hits)